In [1]:
import os
import sys
import json
import textwrap
import subprocess
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
np.random.seed(42)

# ----------------- Config -----------------
DATA_PATH   = "student_dataset_final_preprocessed.csv"
OUT_ROOT    = ""
TOP_N       = 20
MAX_EXPLAIN = 300
BG_SAMPLES  = 150
NSAMPLES    = 100
WF_INSTANCE_REG = 0
WF_INSTANCE_CLS = 0
WATERFALL_ALL_CLASSES = True
WATERFALL_CLASS_NAME = "High Risk"
SEQ_COLS        = ["course1_marks", "course2_marks", "course3_marks", "course4_marks", "course5_marks"]
STATIC_COLS_RAW = ["currentCGPA", "previousCGPA", "semester", "totalcredithours", "attendanceRate", "studyHours"]
BASE_FEATURES   = SEQ_COLS + STATIC_COLS_RAW

outdir = os.path.join(
    OUT_ROOT,
    f"shap_ann_cnn_kernel_SAFE_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
)
os.makedirs(outdir, exist_ok=True)
print("Saving to:", outdir)

train_script_path = os.path.join(outdir, "train_ann_cnn_safe.py")

train_code = f"""
import os
import json
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    r2_score, mean_absolute_error, accuracy_score,
    f1_score, mean_squared_error
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")
np.random.seed(42)
tf.random.set_seed(42)

DATA_PATH = r"{DATA_PATH}"
OUTDIR    = r"{outdir}"
BASE_FEAT_ORIG = {BASE_FEATURES}

def norm_col(c: str) -> str:
    c = str(c).strip()
    c = c.replace(" ", "").replace("-", "_")
    return c.lower()

assert os.path.exists(DATA_PATH), f"Dataset not found: {{DATA_PATH}}"
df = pd.read_csv(DATA_PATH)

# Normalize columns
df.columns = [norm_col(c) for c in df.columns]
BASE_FEAT = [norm_col(c) for c in BASE_FEAT_ORIG]

# -------------------------
# Detect targets robustly
# -------------------------
target_candidates_reg = [
    "next_semester_cgpa",
    "nextsemester_cgpa",
    "next_semestercgpa",
    "next_cgpa",
    "nextcgpa"
]
target_candidates_cls = [
    "recommendation",
    "reco",
    "risk",
    "risk_class",
    "performance_class"
]

reg_target = None
for t in target_candidates_reg:
    if t in df.columns:
        reg_target = t
        break

cls_target = None
for t in target_candidates_cls:
    if t in df.columns:
        cls_target = t
        break

meta = {{
    "reg_target": reg_target,
    "cls_target": cls_target
}}
with open(os.path.join(OUTDIR, "present_targets.json"), "w") as f:
    json.dump(meta, f, indent=2)

missing = [c for c in BASE_FEAT if c not in df.columns]
if missing:
    raise ValueError(
        "Missing feature columns after normalization: " + ", ".join(missing) +
        "\\nAvailable columns sample: " + ", ".join(df.columns[:80])
    )

Xraw = df[BASE_FEAT].copy()
for c in Xraw.columns:
    Xraw[c] = pd.to_numeric(Xraw[c], errors="coerce")
    Xraw[c] = Xraw[c].fillna(Xraw[c].median())

def to_ann_cnn_inputs(raw_df):
    X = raw_df[BASE_FEAT].values.astype(float)
    cnn = X.reshape(-1, len(BASE_FEAT), 1).astype(np.float32)
    ann = X.astype(np.float32)
    return cnn, ann

def make_ann_cnn_reg(n_features):
    cnn_inp = keras.Input(shape=(n_features, 1), name="cnn_input")
    c = layers.Conv1D(64, kernel_size=3, activation="relu")(cnn_inp)
    c = layers.MaxPooling1D(pool_size=2)(c)
    c = layers.Dropout(0.25)(c)
    c = layers.Flatten()(c)
    c = layers.Dense(64, activation="relu")(c)

    ann_inp = keras.Input(shape=(n_features,), name="ann_input")
    a = layers.Dense(128, activation="relu")(ann_inp)
    a = layers.Dropout(0.30)(a)
    a = layers.Dense(64, activation="relu")(a)

    merged = layers.Concatenate()([c, a])
    shared = layers.Dense(96, activation="relu")(merged)
    shared = layers.Dropout(0.30)(shared)

    out = layers.Dense(1, activation="linear", name="cgpa_output")(shared)

    model = keras.Model([cnn_inp, ann_inp], out, name="ANN_CNN_REG")
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model

def make_ann_cnn_cls(n_features, n_classes):
    cnn_inp = keras.Input(shape=(n_features, 1), name="cnn_input")
    c = layers.Conv1D(64, kernel_size=3, activation="relu")(cnn_inp)
    c = layers.MaxPooling1D(pool_size=2)(c)
    c = layers.Dropout(0.25)(c)
    c = layers.Flatten()(c)
    c = layers.Dense(64, activation="relu")(c)

    ann_inp = keras.Input(shape=(n_features,), name="ann_input")
    a = layers.Dense(128, activation="relu")(ann_inp)
    a = layers.Dropout(0.30)(a)
    a = layers.Dense(64, activation="relu")(a)

    merged = layers.Concatenate()([c, a])
    shared = layers.Dense(96, activation="relu")(merged)
    shared = layers.Dropout(0.30)(shared)

    out = layers.Dense(n_classes, activation="softmax", name="reco_output")(shared)

    model = keras.Model([cnn_inp, ann_inp], out, name="ANN_CNN_CLS")
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return model

if reg_target is not None:
    y = pd.to_numeric(df[reg_target], errors="coerce")
    y = y.fillna(y.median()).values.astype(float)

    Xtr_raw, Xte_raw, ytr, yte = train_test_split(
        Xraw, y, test_size=0.2, random_state=42
    )

    Xcnn_tr, Xann_tr = to_ann_cnn_inputs(Xtr_raw)
    Xcnn_te, Xann_te = to_ann_cnn_inputs(Xte_raw)

    reg = make_ann_cnn_reg(n_features=len(BASE_FEAT))
    cb = keras.callbacks.EarlyStopping(
        patience=5, monitor="val_loss", restore_best_weights=True
    )

    reg.fit(
        [Xcnn_tr, Xann_tr], ytr,
        validation_split=0.2,
        epochs=50,
        batch_size=64,
        verbose=0,
        callbacks=[cb]
    )

    pred = reg.predict([Xcnn_te, Xann_te], verbose=0).ravel()

    mse = float(mean_squared_error(yte, pred))
    rmse = float(np.sqrt(mse))

    reg_metrics = {{
        "R2": float(r2_score(yte, pred)),
        "MAE": float(mean_absolute_error(yte, pred)),
        "MSE": mse,
        "RMSE": rmse,
        "reg_target": reg_target
    }}

    with open(os.path.join(OUTDIR, "regression_metrics.json"), "w") as f:
        json.dump(reg_metrics, f, indent=2)

    reg.save(os.path.join(OUTDIR, "reg_model.keras"))

if cls_target is not None:
    y_raw = df[cls_target].astype(str).fillna("Unknown")
    classes = sorted(y_raw.unique().tolist())
    class_to_idx = {{c: i for i, c in enumerate(classes)}}
    y = y_raw.map(class_to_idx).astype(int).values
    n_classes = len(classes)

    try:
        Xtr_raw_c, Xte_raw_c, ytr_idx, yte_idx = train_test_split(
            Xraw, y, test_size=0.2, random_state=42, stratify=y
        )
        strat_used = True
    except Exception:
        Xtr_raw_c, Xte_raw_c, ytr_idx, yte_idx = train_test_split(
            Xraw, y, test_size=0.2, random_state=42
        )
        strat_used = False

    Xcnn_tr_c, Xann_tr_c = to_ann_cnn_inputs(Xtr_raw_c)
    Xcnn_te_c, Xann_te_c = to_ann_cnn_inputs(Xte_raw_c)

    ytr_oh = keras.utils.to_categorical(ytr_idx, num_classes=n_classes)
    yte_oh = keras.utils.to_categorical(yte_idx, num_classes=n_classes)

    cls = make_ann_cnn_cls(n_features=len(BASE_FEAT), n_classes=n_classes)
    cb = keras.callbacks.EarlyStopping(
        patience=5, monitor="val_loss", restore_best_weights=True
    )

    cls.fit(
        [Xcnn_tr_c, Xann_tr_c], ytr_oh,
        validation_split=0.2,
        epochs=60,
        batch_size=64,
        verbose=0,
        callbacks=[cb]
    )

    proba = cls.predict([Xcnn_te_c, Xann_te_c], verbose=0)
    yhat = np.argmax(proba, axis=1)

    mse_proba = float(np.mean((yte_oh - proba) ** 2))
    rmse_proba = float(np.sqrt(mse_proba))

    cls_metrics = {{
        "Accuracy": float(accuracy_score(yte_idx, yhat)),
        "F1_macro": float(f1_score(yte_idx, yhat, average="macro")),
        "MSE_proba": mse_proba,
        "RMSE_proba": rmse_proba,
        "labels": classes,
        "cls_target": cls_target,
        "stratified_split": strat_used
    }}

    with open(os.path.join(OUTDIR, "classification_metrics.json"), "w") as f:
        json.dump(cls_metrics, f, indent=2)

    cls.save(os.path.join(OUTDIR, "cls_model.keras"))
    with open(os.path.join(OUTDIR, "classes.json"), "w") as f:
        json.dump(classes, f, indent=2)

print("TRAINING_DONE")
"""

with open(train_script_path, "w", encoding="utf-8") as f:
    f.write(textwrap.dedent(train_code))

print("Training in a clean subprocess...")

proc = subprocess.run(
    [sys.executable, train_script_path],
    capture_output=True,
    text=True
)

with open(os.path.join(outdir, "TRAIN_STDOUT.txt"), "w", encoding="utf-8") as f:
    f.write(proc.stdout)

with open(os.path.join(outdir, "TRAIN_STDERR.txt"), "w", encoding="utf-8") as f:
    f.write(proc.stderr)

print("\n--- TRAIN STDOUT (head) ---\n", proc.stdout[:4000])
print("\n--- TRAIN STDERR (head) ---\n", proc.stderr[:4000])

if proc.returncode != 0:
    raise RuntimeError(
        f"Training subprocess failed (code={proc.returncode}). "
        f"Check:\n"
        f"- {os.path.join(outdir, 'TRAIN_STDOUT.txt')}\n"
        f"- {os.path.join(outdir, 'TRAIN_STDERR.txt')}"
    )

print("Training finished.\n")

import shap

if not hasattr(np, "bool"):
    np.bool = np.bool_

def norm_col(c: str) -> str:
    c = str(c).strip()
    c = c.replace(" ", "").replace("-", "_")
    return c.lower()

def bar_plot(names, values, title, outfile):
    plt.figure()
    idx = np.arange(len(names))
    plt.bar(idx, values)
    plt.xticks(idx, names, rotation=90)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(outfile, dpi=200, bbox_inches="tight")
    plt.close()

def counts_plot(names, pos, neg, title, outfile):
    plt.figure()
    idx = np.arange(len(names))
    w = 0.45
    plt.bar(idx - w/2, pos, w, label="Positive")
    plt.bar(idx + w/2, neg, w, label="Negative")
    plt.xticks(idx, names, rotation=90)
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(outfile, dpi=200, bbox_inches="tight")
    plt.close()

def beeswarm(exp, title, outfile, max_display=TOP_N):
    plt.figure(figsize=(7, 6))
    shap.plots.beeswarm(exp, max_display=max_display, show=False)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(outfile, dpi=200, bbox_inches="tight")
    plt.close()

def dependence_plot(exp, feature, color_feature, title, outfile):
    plt.figure(figsize=(7, 5))
    try:
        shap.plots.scatter(exp[:, feature], color=exp[:, color_feature], show=False)
    except Exception:
        shap.dependence_plot(
            feature,
            exp.values,
            exp.data,
            feature_names=exp.feature_names,
            interaction_index=color_feature,
            show=False
        )
    plt.title(title)
    plt.tight_layout()
    plt.savefig(outfile, dpi=200, bbox_inches="tight")
    plt.close()

def waterfall_plot(exp_single, title, outfile, max_display=TOP_N):
    plt.figure(figsize=(8, 6))
    shap.plots.waterfall(exp_single, max_display=max_display, show=False)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(outfile, dpi=200, bbox_inches="tight")
    plt.close()

def table_and_counts_from_values(sv, feature_names, top_n, prefix, outdir):
    sv = np.asarray(sv)
    mean_abs = np.mean(np.abs(sv), axis=0)
    mean_val = np.mean(sv, axis=0)

    df_imp = pd.DataFrame({
        "feature": feature_names,
        "mean_abs_shap": mean_abs,
        "mean_shap": mean_val
    }).sort_values("mean_abs_shap", ascending=False)

    imp_csv = os.path.join(outdir, f"{prefix}_feature_importance.csv")
    df_imp.to_csv(imp_csv, index=False)

    top = df_imp.head(top_n)
    bar_plot(
        top["feature"].tolist(),
        top["mean_abs_shap"].values,
        f"Top {top_n} Feature Importance (mean |SHAP|) — {prefix}",
        os.path.join(outdir, f"{prefix}_top{top_n}_mean_abs_shap.png")
    )

    top_idx = [feature_names.index(f) for f in top["feature"].tolist()]
    sv_top = sv[:, top_idx]
    pos = np.sum(sv_top > 0, axis=0)
    neg = np.sum(sv_top < 0, axis=0)

    df_cnt = pd.DataFrame({
        "feature": top["feature"].tolist(),
        "positive_count": pos,
        "negative_count": neg
    })

    cnt_csv = os.path.join(outdir, f"{prefix}_shap_counts_top{top_n}.csv")
    df_cnt.to_csv(cnt_csv, index=False)

    counts_plot(
        df_cnt["feature"].tolist(),
        df_cnt["positive_count"].values,
        df_cnt["negative_count"].values,
        f"Counts of ± SHAP (Top {top_n}) — {prefix}",
        os.path.join(outdir, f"{prefix}_top{top_n}_pos_neg_counts.png")
    )
    return imp_csv, cnt_csv

assert os.path.exists(DATA_PATH), "Dataset missing."
df = pd.read_csv(DATA_PATH)
df.columns = [norm_col(c) for c in df.columns]

meta = json.load(open(os.path.join(outdir, "present_targets.json"), "r"))
reg_target = meta.get("reg_target")
cls_target = meta.get("cls_target")

BASE_FEATURES_N = [norm_col(c) for c in BASE_FEATURES]

Xraw = df[BASE_FEATURES_N].copy()
for c in Xraw.columns:
    Xraw[c] = pd.to_numeric(Xraw[c], errors="coerce")
    Xraw[c] = Xraw[c].fillna(Xraw[c].median())

if reg_target is not None:
    from tensorflow import keras
    from sklearn.model_selection import train_test_split

    y_reg = pd.to_numeric(df[reg_target], errors="coerce")
    y_reg = y_reg.fillna(y_reg.median()).values.astype(float)

    Xtr_raw, Xte_raw, _, _ = train_test_split(
        Xraw, y_reg, test_size=0.2, random_state=42
    )

    reg_model = keras.models.load_model(
        os.path.join(outdir, "reg_model.keras"),
        compile=False
    )

    def predict_reg_on_raw(df_like):
        D = pd.DataFrame(df_like, columns=BASE_FEATURES_N)
        X = D[BASE_FEATURES_N].values.astype(float)
        cnn = X.reshape(-1, len(BASE_FEATURES_N), 1)
        ann = X
        return reg_model.predict([cnn, ann], verbose=0).ravel()

    bg_reg = Xtr_raw.sample(min(BG_SAMPLES, len(Xtr_raw)), random_state=42)
    Xe_reg = Xte_raw.sample(min(MAX_EXPLAIN, len(Xte_raw)), random_state=42)

    expl_reg = shap.Explainer(
        predict_reg_on_raw,
        masker=shap.maskers.Independent(bg_reg)
    )
    exp_reg = expl_reg(Xe_reg, max_evals=NSAMPLES)

    beeswarm(
        exp_reg,
        f"SHAP Beeswarm — {reg_target}",
        os.path.join(outdir, f"reg_{reg_target}_beeswarm.png"),
        max_display=TOP_N
    )

    table_and_counts_from_values(
        exp_reg.values,
        list(Xe_reg.columns),
        TOP_N,
        f"reg_{reg_target}",
        outdir
    )

    if "currentcgpa" in Xe_reg.columns and "studyhours" in Xe_reg.columns:
        dependence_plot(
            exp_reg,
            feature="currentcgpa",
            color_feature="studyhours",
            title="Dependence — currentcgpa (color: studyhours) — Regression",
            outfile=os.path.join(outdir, "reg_dependence_currentcgpa_color_studyhours.png")
        )

    wf_idx = max(0, min(int(WF_INSTANCE_REG), len(Xe_reg) - 1))
    waterfall_plot(
        exp_reg[wf_idx],
        title=f"Waterfall — {reg_target} — instance {wf_idx}",
        outfile=os.path.join(outdir, f"reg_waterfall_{reg_target}_instance_{wf_idx}.png"),
        max_display=TOP_N
    )

if cls_target is not None:
    from tensorflow import keras
    from sklearn.model_selection import train_test_split

    with open(os.path.join(outdir, "classes.json"), "r") as f:
        classes = json.load(f)

    cls_model = keras.models.load_model(
        os.path.join(outdir, "cls_model.keras"),
        compile=False
    )

    y_raw = df[cls_target].astype(str).fillna("Unknown")
    class_to_idx = {c: i for i, c in enumerate(classes)}
    y_enc = y_raw.map(class_to_idx).astype(int).values

    try:
        Xtr_raw_c, Xte_raw_c, _, _ = train_test_split(
            Xraw, y_enc, test_size=0.2, random_state=42, stratify=y_enc
        )
    except Exception:
        Xtr_raw_c, Xte_raw_c, _, _ = train_test_split(
            Xraw, y_enc, test_size=0.2, random_state=42
        )

    def predict_proba_on_raw(df_like):
        D = pd.DataFrame(df_like, columns=BASE_FEATURES_N)
        X = D[BASE_FEATURES_N].values.astype(float)
        cnn = X.reshape(-1, len(BASE_FEATURES_N), 1)
        ann = X
        return cls_model.predict([cnn, ann], verbose=0)

    bg_cls = Xtr_raw_c.sample(min(BG_SAMPLES, len(Xtr_raw_c)), random_state=42)
    Xe_cls = Xte_raw_c.sample(min(MAX_EXPLAIN, len(Xte_raw_c)), random_state=42)

    expl_cls = shap.Explainer(
        predict_proba_on_raw,
        masker=shap.maskers.Independent(bg_cls)
    )
    exp_cls = expl_cls(Xe_cls, max_evals=NSAMPLES)

    vals = exp_cls.values
    if np.ndim(vals) == 2:
        vals = vals[:, :, None]

    feat_names = list(Xe_cls.columns)
    Kc = vals.shape[2]
    base_vals_arr = np.array(exp_cls.base_values)

    for ci in range(Kc):
        cname = classes[ci] if ci < len(classes) else f"class_{ci}"

        exp_c = shap.Explanation(
            values=vals[:, :, ci],
            base_values=(base_vals_arr[:, ci] if base_vals_arr.ndim == 2 else base_vals_arr),
            data=np.array(exp_cls.data),
            feature_names=feat_names
        )

        beeswarm(
            exp_c,
            f"SHAP Beeswarm — {cls_target} — {cname}",
            os.path.join(outdir, f"clf_{cls_target}_beeswarm_{cname}.png"),
            max_display=TOP_N
        )

        table_and_counts_from_values(
            exp_c.values,
            feat_names,
            TOP_N,
            f"clf_{cls_target}_{cname}",
            outdir
        )

        if str(cname).lower() == "dropout":
            if "studyhours" in feat_names and "currentcgpa" in feat_names:
                dependence_plot(
                    exp_c,
                    feature="studyhours",
                    color_feature="currentcgpa",
                    title="Dependence — studyhours (color: currentcgpa) — Dropout",
                    outfile=os.path.join(outdir, "clf_dependence_studyhours_color_currentcgpa_Dropout.png")
                )

        do_wf = WATERFALL_ALL_CLASSES or (str(cname).lower() == str(WATERFALL_CLASS_NAME).lower())
        if do_wf:
            wf_idx = max(0, min(int(WF_INSTANCE_CLS), len(Xe_cls) - 1))
            waterfall_plot(
                exp_c[wf_idx],
                title=f"Waterfall — {cls_target} — {cname} — instance {wf_idx}",
                outfile=os.path.join(outdir, f"clf_waterfall_{cls_target}_{cname}_instance_{wf_idx}.png"),
                max_display=TOP_N
            )

    mean_abs_sum = np.sum(
        [np.mean(np.abs(vals[:, :, ci]), axis=0) for ci in range(Kc)],
        axis=0
    )

    df_overall = pd.DataFrame({
        "feature": feat_names,
        "mean_abs_shap_across_classes": mean_abs_sum
    }).sort_values("mean_abs_shap_across_classes", ascending=False)

    df_overall.to_csv(
        os.path.join(outdir, f"clf_{cls_target}_feature_importance_overall.csv"),
        index=False
    )

    top = df_overall.head(TOP_N)
    bar_plot(
        top["feature"].tolist(),
        top["mean_abs_shap_across_classes"].values,
        f"Top {TOP_N} Overall Feature Importance — {cls_target}",
        os.path.join(outdir, f"clf_{cls_target}_top{TOP_N}_overall_mean_abs_shap.png")
    )

print("\\nDONE. All artifacts saved under:", outdir)

Saving to: shap_ann_cnn_kernel_SAFE_20260322_153239
Training in a clean subprocess...

--- TRAIN STDOUT (head) ---
 TRAINING_DONE


--- TRAIN STDERR (head) ---
 2026-03-22 15:32:41.185892: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-22 15:32:41.190731: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-22 15:32:41.204121: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774193561.227999    3827 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774193561.235022    3827 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has alre

PermutationExplainer explainer: 301it [05:32,  1.13s/it]
PermutationExplainer explainer: 301it [06:44,  1.39s/it]


\nDONE. All artifacts saved under: shap_ann_cnn_kernel_SAFE_20260322_153239


<Figure size 700x500 with 0 Axes>

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from tensorflow import keras
from sklearn.model_selection import train_test_split

np.random.seed(42)

# Config
DATA_PATH = "student_dataset_final_preprocessed.csv"
TOP_N = 20

# Load dataset
df = pd.read_csv(DATA_PATH)

# Normalize column names
df.columns = [c.strip().replace(" ", "").replace("-", "_").lower() for c in df.columns]

# Features
FEATURES = [
    "course1_marks","course2_marks","course3_marks",
    "course4_marks","course5_marks",
    "currentcgpa","previouscgpa","semester",
    "totalcredithours","attendancerate","studyhours"
]

X = df[FEATURES].copy()

y_reg = df["next_semester_cgpa"]
y_cls = df["recommendation"]

X_train, X_test, y_reg_train, y_reg_test, y_cls_train, y_cls_test = train_test_split(
    X, y_reg, y_cls, test_size=0.2, random_state=42
)

def to_inputs(X):
    arr = X.values.astype(np.float32)
    cnn = arr.reshape(-1, arr.shape[1], 1)
    ann = arr
    return cnn, ann

Xcnn_train, Xann_train = to_inputs(X_train)
Xcnn_test, Xann_test = to_inputs(X_test)

def build_reg_model(n_features):
    cnn_in = keras.Input(shape=(n_features,1))
    c = keras.layers.Conv1D(64,3,activation="relu")(cnn_in)
    c = keras.layers.MaxPooling1D(2)(c)
    c = keras.layers.Flatten()(c)

    ann_in = keras.Input(shape=(n_features,))
    a = keras.layers.Dense(128,activation="relu")(ann_in)

    x = keras.layers.Concatenate()([c,a])
    x = keras.layers.Dense(64,activation="relu")(x)

    out = keras.layers.Dense(1)(x)

    model = keras.Model([cnn_in, ann_in], out)
    model.compile(optimizer="adam", loss="mse")
    return model

model_reg = build_reg_model(len(FEATURES))
model_reg.fit(
    [Xcnn_train, Xann_train],
    y_reg_train,
    epochs=10,
    batch_size=32,
    verbose=1
)

def predict_reg(df_input):
    X = df_input.values.astype(np.float32)
    return model_reg.predict([X.reshape(-1, X.shape[1],1), X], verbose=0).ravel()

explainer = shap.Explainer(predict_reg, X_train)
exp = explainer(X_test)

def waterfall_plot(exp_single, title, outfile):
    plt.figure(figsize=(9,6))
    shap.plots.waterfall(exp_single, show=False)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(outfile, dpi=300)
    plt.close()

wf_idx = 0
exp_single = exp[wf_idx]

sorted_idx = np.argsort(np.abs(exp_single.values))[::-1]

exp_sorted = shap.Explanation(
    values=exp_single.values[sorted_idx],
    base_values=exp_single.base_values,
    data=exp_single.data[sorted_idx],
    feature_names=[exp_single.feature_names[i] for i in sorted_idx]
)

waterfall_plot(
    exp_sorted,
    "Waterfall — ANN+CNN Regression",
    "reg_waterfall_sorted.png"
)

from tensorflow.keras.utils import to_categorical

y_cls_cat = to_categorical(y_cls_train)

def build_cls_model(n_features, n_classes):
    cnn_in = keras.Input(shape=(n_features,1))
    c = keras.layers.Conv1D(64,3,activation="relu")(cnn_in)
    c = keras.layers.MaxPooling1D(2)(c)
    c = keras.layers.Flatten()(c)

    ann_in = keras.Input(shape=(n_features,))
    a = keras.layers.Dense(128,activation="relu")(ann_in)

    x = keras.layers.Concatenate()([c,a])
    x = keras.layers.Dense(64,activation="relu")(x)

    out = keras.layers.Dense(n_classes,activation="softmax")(x)

    model = keras.Model([cnn_in, ann_in], out)
    model.compile(optimizer="adam", loss="categorical_crossentropy")
    return model

n_classes = len(np.unique(y_cls))
model_cls = build_cls_model(len(FEATURES), n_classes)

model_cls.fit(
    [Xcnn_train, Xann_train],
    y_cls_cat,
    epochs=10,
    batch_size=32,
    verbose=1
)

def predict_cls(df_input):
    X = df_input.values.astype(np.float32)
    return model_cls.predict([X.reshape(-1, X.shape[1],1), X], verbose=0)

explainer_cls = shap.Explainer(predict_cls, X_train)
exp_cls = explainer_cls(X_test)

def dependence_plot(exp, feature, color_feature, outfile):
    plt.figure(figsize=(7,5))
    shap.plots.scatter(exp[:, feature], color=exp[:, color_feature], show=False)
    plt.tight_layout()
    plt.savefig(outfile, dpi=300)
    plt.close()

# Assume "Dropout" is class index 4 (adjust if needed)
dropout_class_index = 4

exp_dropout = shap.Explanation(
    values=exp_cls.values[:,:,dropout_class_index],
    base_values=exp_cls.base_values[:,dropout_class_index],
    data=exp_cls.data,
    feature_names=FEATURES
)

if "studyhours" in FEATURES and "currentcgpa" in FEATURES:
    dependence_plot(
        exp_dropout,
        feature="studyhours",
        color_feature="currentcgpa",
        outfile="dependence_dropout.png"
    )

print("Waterfall + Dependence plots generated")

Epoch 1/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.7376
Epoch 2/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.1733
Epoch 3/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.1333
Epoch 4/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.1076
Epoch 5/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0900
Epoch 6/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0770
Epoch 7/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0652
Epoch 8/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0554
Epoch 9/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0471
Epoch 10/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0416


PermutationExplainer explainer:   3%|▎         | 11/334 [00:49<25:08,  4.67s/it]

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# config
INPUT_CSV = "reg_next_semester_CGPA_feature_importance.csv"
TOP_N     = 15
OUT_DIR   = os.path.dirname(INPUT_CSV) or "."

#load & pick top-N
df = pd.read_csv(INPUT_CSV)
if not {"feature","mean_abs_shap"}.issubset(df.columns):
    raise ValueError("CSV must contain columns: feature, mean_abs_shap (and ideally mean_shap).")

top = df.sort_values("mean_abs_shap", ascending=False).head(TOP_N).reset_index(drop=True)

plt.figure(figsize=(9, 4.5))
plt.bar(range(len(top)), top["mean_abs_shap"].values)
plt.xticks(range(len(top)), top["feature"].tolist(), rotation=90)
plt.title(f"Top {TOP_N} Feature Importance — Regression (mean |SHAP|)")
plt.tight_layout()
bar_path = os.path.join(OUT_DIR, "reg_top15_mean_abs_shap.png")
plt.savefig(bar_path, dpi=200, bbox_inches="tight")
plt.close()
print("Saved:", bar_path)

if "mean_shap" not in df.columns:
    print("mean_shap column not found; skipping direction table.")
else:
    summary = top[["feature","mean_abs_shap","mean_shap"]].copy()
    summary["direction"] = np.where(
        summary["mean_shap"] > 0, "↑ raises prediction",
        np.where(summary["mean_shap"] < 0, "↓ lowers prediction", "≈ little net effect")
    )

    fig_h = 0.45 * (len(summary) + 2)
    plt.figure(figsize=(9, max(3.5, fig_h)))
    plt.axis("off")
    tbl = plt.table(
        cellText=summary.round(4).values,
        colLabels=summary.columns.tolist(),
        loc="center",
        cellLoc="center",
        colLoc="center",
    )
    tbl.scale(1.1, 1.25)

    dir_col_idx = summary.columns.get_loc("direction")
    for r in range(1, len(summary) + 1):
        cell = tbl[(r, dir_col_idx)]
        text = cell.get_text().get_text()
        if text.startswith("↑"):
            cell.set_facecolor("#3CB371")
            cell.set_text_props(color="white", weight="bold")
        elif text.startswith("↓"):
            cell.set_facecolor("#DC143C")
            cell.set_text_props(color="white", weight="bold")

    plt.title("Regression — Top 15 with mean SHAP direction")
    table_path = os.path.join(OUT_DIR, "reg_top15_mean_shap_direction_table.png")
    plt.savefig(table_path, dpi=200, bbox_inches="tight")
    plt.close()
    print("Saved:", table_path)

Saved: .\reg_top15_mean_abs_shap.png
Saved: .\reg_top15_mean_shap_direction_table.png


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# config
INPUT_CSV = "reg_next_semester_cgpa_feature_importance.csv"
TOP_N     = 15
OUT_DIR   = os.path.dirname(INPUT_CSV) or "."

# load
df = pd.read_csv(INPUT_CSV)

if not {"feature", "mean_abs_shap"}.issubset(df.columns):
    raise ValueError("CSV must contain columns: feature, mean_abs_shap, and ideally mean_shap.")

top = (
    df.sort_values("mean_abs_shap", ascending=False)
      .head(TOP_N)
      .reset_index(drop=True)
)

base_name = os.path.basename(INPUT_CSV).lower()

if base_name.startswith("reg_"):
    plot_title = f"Top {TOP_N} Feature Importance — ANN+CNN Regression (mean |SHAP|)"
    output_name = f"anncnn_reg_top{TOP_N}_shap_horizontal_bars.png"
elif base_name.startswith("clf_"):
    plot_title = f"Top {TOP_N} Feature Importance — ANN+CNN Classification (mean |SHAP|)"
    output_name = f"anncnn_clf_top{TOP_N}_shap_horizontal_bars.png"
else:
    plot_title = f"Top {TOP_N} Feature Importance — ANN+CNN (mean |SHAP|)"
    output_name = f"anncnn_top{TOP_N}_shap_horizontal_bars.png"

colors = []

if "mean_shap" in top.columns:
    for s in top["mean_shap"]:
        if s > 0.005:
            colors.append("#4CAF50")
        elif s < -0.005:
            colors.append("#C44E52")
        else:
            colors.append("#4C72B0")
else:
    colors = ["#4C72B0"] * len(top)

plt.figure(figsize=(8, 6))
plt.barh(top["feature"], top["mean_abs_shap"], color=colors)
plt.gca().invert_yaxis()

plt.xlabel("SHAP values (importances)")
plt.title(plot_title)
plt.tight_layout()

out_path = os.path.join(OUT_DIR, output_name)
plt.savefig(out_path, dpi=200, bbox_inches="tight")
plt.close()

print("Saved:", out_path)

Saved: .\anncnn_reg_top15_shap_horizontal_bars.png


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# config
INPUT_CSV = "clf_recommendation_2_feature_importance.csv"
TOP_N     = 15
OUT_DIR   = os.path.dirname(INPUT_CSV) or "."

# load
df = pd.read_csv(INPUT_CSV)

if not {"feature", "mean_abs_shap"}.issubset(df.columns):
    raise ValueError("CSV must contain columns: feature, mean_abs_shap, and ideally mean_shap.")

top = (
    df.sort_values("mean_abs_shap", ascending=False)
      .head(TOP_N)
      .reset_index(drop=True)
)

base_name = os.path.basename(INPUT_CSV).lower()

if base_name.startswith("reg_"):
    plot_title = f"Top {TOP_N} Feature Importance — ANN+CNN Regression (mean |SHAP|)"
    output_name = f"anncnn_reg_top{TOP_N}_shap_horizontal_bars.png"
elif base_name.startswith("clf_"):
    plot_title = f"Top {TOP_N} Feature Importance — ANN+CNN Classification (mean |SHAP|)"
    output_name = f"anncnn_clf_top{TOP_N}_shap_horizontal_bars.png"
else:
    plot_title = f"Top {TOP_N} Feature Importance — ANN+CNN (mean |SHAP|)"
    output_name = f"anncnn_top{TOP_N}_shap_horizontal_bars.png"

colors = []

if "mean_shap" in top.columns:
    for s in top["mean_shap"]:
        if s > 0.005:
            colors.append("#4CAF50")
        elif s < -0.005:
            colors.append("#C44E52")
        else:
            colors.append("#4C72B0")
else:
    colors = ["#4C72B0"] * len(top)

plt.figure(figsize=(8, 6))
plt.barh(top["feature"], top["mean_abs_shap"], color=colors)
plt.gca().invert_yaxis()

plt.xlabel("SHAP values (importances)")
plt.title(plot_title)
plt.tight_layout()

out_path = os.path.join(OUT_DIR, output_name)
plt.savefig(out_path, dpi=200, bbox_inches="tight")
plt.close()

print("Saved:", out_path)

Saved: .\anncnn_clf_top15_shap_horizontal_bars.png


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# config
INPUT_CSV = "reg_next_semester_cgpa_feature_importance.csv"   # change if needed
TOP_N     = 15
OUT_DIR   = os.path.dirname(INPUT_CSV) or "."

# load
df = pd.read_csv(INPUT_CSV)

if not {"feature", "mean_abs_shap"}.issubset(df.columns):
    raise ValueError("CSV must contain columns: feature, mean_abs_shap, and ideally mean_shap.")

top = (
    df.sort_values("mean_abs_shap", ascending=False)
      .head(TOP_N)
      .reset_index(drop=True)
)

base_name = os.path.basename(INPUT_CSV).lower()

if base_name.startswith("reg_"):
    plot_title = f"Top {TOP_N} Feature Importance — ANN+CNN Regression (mean |SHAP|)"
    output_name = f"anncnn_reg_top{TOP_N}_shap_horizontal_bars.png"
elif base_name.startswith("clf_"):
    plot_title = f"Top {TOP_N} Feature Importance — ANN+CNN Classification (mean |SHAP|)"
    output_name = f"anncnn_clf_top{TOP_N}_shap_horizontal_bars.png"
else:
    plot_title = f"Top {TOP_N} Feature Importance — ANN+CNN (mean |SHAP|)"
    output_name = f"anncnn_top{TOP_N}_shap_horizontal_bars.png"

colors = []

if "mean_shap" in top.columns:
    for s in top["mean_shap"]:
        if s > 0.005:
            colors.append("#4CAF50")
        elif s < -0.005:
            colors.append("#C44E52")
        else:
            colors.append("#4C72B0")
else:
    colors = ["#4C72B0"] * len(top)

plt.figure(figsize=(8, 6))
plt.barh(top["feature"], top["mean_abs_shap"], color=colors)
plt.gca().invert_yaxis()

plt.xlabel("SHAP values (importances)")
plt.title(plot_title)
plt.tight_layout()

out_path = os.path.join(OUT_DIR, output_name)
plt.savefig(out_path, dpi=200, bbox_inches="tight")
plt.close()

print("Saved:", out_path)

Saved: .\anncnn_reg_top15_shap_horizontal_bars.png
